# Phase 23: Sequence Data Construction

**Goal:** Standard AI Models (like XGBoost) look at every network packet completely in isolation. But many attacks, like **Brute Force Logins** or **Lateral Movement**, require the AI to look at the **entire sequence of events** to realize what is happening.

In this Phase, we will transform our flat 2D Tables into **3D Deep Learning Tensors** so we can feed them into advanced Neural Networks like **LSTMs** and **Transformers**!

In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import math

np.set_printoptions(suppress=True)
torch.manual_seed(42)

### Step 1: Sequence Builder Implementation (Subphase 23.1)
We must convert our `(Samples, Features)` table into a 3D Tensor shaped `(Samples, Window_Size, Features)`.

**The Critical Rule:** We must NEVER mix packets from different IP Addresses into the same sequence! We will strictly group by `source_ip` and use a Sliding Window to build the history. If an IP doesn't have enough history, we mathematically pad the beginning with `0.0`.

In [2]:
class SequenceBuilder:
    def __init__(self, window_size: int = 5, pad_value: float = 0.0):
        self.window_size = window_size
        self.pad_value = pad_value

    def build(self, X: np.ndarray, y: np.ndarray, source_ips: np.ndarray, timestamps: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        # 1. Sort strictly by Source IP, and then chronologically by Timestamp
        sort_idx = np.lexsort((timestamps, source_ips))
        X_sorted, y_sorted, ip_sorted = X[sort_idx], y[sort_idx], source_ips[sort_idx]
        
        # 2. Iterate through every unique user and build their sliding window history
        unique_ips, counts = np.unique(ip_sorted, return_counts=True)
        
        sequences, labels = [], []
        start_idx = 0
        
        for count in counts:
            end_idx = start_idx + count
            ip_X = X_sorted[start_idx:end_idx]
            ip_y = y_sorted[start_idx:end_idx]
            
            # Sliding window per user
            for i in range(count):
                window_end = i + 1
                window_start = max(0, window_end - self.window_size)
                seq = ip_X[window_start:window_end]
                
                # Pad the beginning if they just connected and have no history
                if len(seq) < self.window_size:
                    pad_len = self.window_size - len(seq)
                    padding = np.full((pad_len, X.shape[1]), self.pad_value)
                    seq = np.vstack([padding, seq])
                    
                sequences.append(seq)
                labels.append(ip_y[window_end - 1]) # The label is the final packet in the sequence
                
            start_idx = end_idx
            
        return np.array(sequences), np.array(labels)

print("✅ SequenceBuilder Logic Completed!")

✅ SequenceBuilder Logic Completed!


### Step 2: PyTorch Datasets & DataLoaders (Subphase 23.2)
Deep Learning models are trained on GPUs using `PyTorch`. PyTorch requires strict memory management via `Dataset` and `DataLoader` classes to shuffle and batch the sequences efficiently.

In [3]:
class SequenceDataset(Dataset):
    def __init__(self, X_seq: np.ndarray, y: np.ndarray):
        # Strictly cast to Float32 and Long to prevent PyTorch C++ crashes
        self.X = torch.tensor(X_seq, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def create_dataloader(X_seq, y, batch_size=32, shuffle=True):
    dataset = SequenceDataset(X_seq, y)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

print("✅ PyTorch Dataloaders Complete!")

✅ PyTorch Dataloaders Complete!


### Step 3: Sinusoidal Positional Encoding (Subphase 23.3)
If you pass a sequence of 10 events into a Transformer Neural Network, the Transformer looks at all 10 events simultaneously. It doesn't know which event happened first!

We use the legendary **Vaswani et al. (2017)** mathematics to encode a `sin/cos` mathematical fingerprint directly into the data, forcing the Transformer to understand the concept of linear time!

In [4]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Calculate the mathematical positional frequencies
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        
        # Register as a buffer (It is frozen math, not an AI weight to be updated!)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # Inject the "Time Fingerprint" directly into the network data
        x = x + self.pe[:x.size(0)]
        return self.dropout(x)

print("✅ Transformer Positional Encoding Registered!")

✅ Transformer Positional Encoding Registered!


### Step 4: Sequence Integration Test
Let's put this entirely together. We will create a fake chronological dataset containing two distinct IP Addresses, run it through the SequenceBuilder, and inject it into PyTorch!

In [5]:
# 1. Fake Dataset (5 events for IP_A, 2 events for IP_B)
X = np.array([
    [10, 20], [11, 21], [12, 22], [13, 23], [14, 24], # IP A data
    [99, 99], [88, 88]                                # IP B data
])
y = np.array([0, 0, 0, 1, 1, 0, 0]) # IP A turns malicious at step 4!
ips = np.array(['IP_A', 'IP_A', 'IP_A', 'IP_A', 'IP_A', 'IP_B', 'IP_B'])
times = np.array([1, 2, 3, 4, 5, 1, 2])

# 2. Build 3D Sequences (Window Size = 3)
builder = SequenceBuilder(window_size=3)
X_seq, y_seq = builder.build(X, y, ips, times)

print(f"\n=== 3D SEQUENCE TENSOR ===")
print(f"Original Tabular Shape: {X.shape}")
print(f"Deep Learning Sequence Shape: {X_seq.shape}")

print("\nExample output of IP_B's final sequence:")
print(X_seq[-1])
print("(Notice how it safely zero-padded the beginning because IP_B only had 2 events, preventing it from accidentally pulling in IP_A's data!)")

# 3. Run it through the PyTorch Dataloader
loader = create_dataloader(X_seq, y_seq, batch_size=2)
batch_X, batch_y = next(iter(loader))

print(f"\n=== PYTORCH INTEGRATION ===")
print(f"Batch Shape: {batch_X.shape}")
print(f"Batch Data Type: {batch_X.dtype}")
print("✅ SUCCESS: The dataset is officially ready for LSTM / Transformer Training!")


=== 3D SEQUENCE TENSOR ===
Original Tabular Shape: (7, 2)
Deep Learning Sequence Shape: (7, 3, 2)

Example output of IP_B's final sequence:
[[ 0.  0.]
 [99. 99.]
 [88. 88.]]
(Notice how it safely zero-padded the beginning because IP_B only had 2 events, preventing it from accidentally pulling in IP_A's data!)

=== PYTORCH INTEGRATION ===
Batch Shape: torch.Size([2, 3, 2])
Batch Data Type: torch.float32
✅ SUCCESS: The dataset is officially ready for LSTM / Transformer Training!
